# 01 — Data audit & exploratory analysis

**Decision question:** tại snapshot đầu tháng, khách hàng nào có nguy cơ rời mạng trong 30 ngày tới?  \n**Unit of analysis:** một khách hàng. **Target:** `Churn=Yes`. **Primary downstream metric:** Average Precision.

Notebook này là lớp trình bày có thể tái lập; logic kiểm tra dữ liệu nằm trong `src/telco_churn/eda.py` để CLI, test và notebook dùng chung một nguồn sự thật. Các quan hệ quan sát được không được diễn giải là quan hệ nhân quả.

## 0. Setup và reproducibility contract

Chạy notebook từ repository root sau `python -m pip install -e ".[dev]"`. Raw data phải đúng SHA-256 đã review; notebook dừng ngay khi file hoặc schema thay đổi.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from telco_churn.eda import (
    categorical_summary,
    load_and_clean,
    numeric_summary,
    schema_profile,
    sha256_file,
)
from telco_churn.schema import DATA_SHA256

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DATA_PATH = ROOT / "data/raw/Telco-Customer-Churn.csv"
assert DATA_PATH.exists(), "Run: python -m telco_churn.data fetch"
assert sha256_file(DATA_PATH) == DATA_SHA256
pd.set_option("display.max_columns", 50)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load, parse và kiểm tra chất lượng

`TotalCharges` được parse sang numeric. 11 chuỗi trống hợp lệ theo điều kiện đã audit (`tenure=0`) được giữ là missing và sẽ được impute **bên trong CV pipeline**, không điền trước khi split.

In [2]:
data, quality = load_and_clean(DATA_PATH)
display(pd.Series(quality, name="value").to_frame())
assert quality["rows"] == 7_043
assert quality["duplicate_customer_ids"] == 0
assert quality["blank_total_charges"] == 11
assert quality["blank_total_charges_with_tenure_zero"] == 11
display(data.head(3))

,value
rows,7043
columns_raw,21
sha256,16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd...
source_url,https://raw.githubusercontent.com/IBM/telco-cu...
duplicate_rows,0
duplicate_customer_ids,0
blank_total_charges,11
numeric_missing_total_charges_after_parse,11
blank_total_charges_with_tenure_zero,11
invalid_churn_labels,0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,churn_flag
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1


In [3]:
profile = schema_profile(data)
display(profile)
display(profile.loc[profile["missing"].gt(0)])

,column,dtype_after_load,non_null,missing,unique_non_null,min,max,sample_values
0,customerID,object,7043,0,7043,,,7590-VHVEG | 5575-GNVDE | 3668-QPYBK | 7795-CF...
1,gender,object,7043,0,2,,,Female | Male
2,SeniorCitizen,int64,7043,0,2,0,1,0 | 1
3,Partner,object,7043,0,2,,,Yes | No
4,Dependents,object,7043,0,2,,,No | Yes
5,tenure,int64,7043,0,73,0,72,1 | 34 | 2 | 45 | 8
6,PhoneService,object,7043,0,2,,,No | Yes
7,MultipleLines,object,7043,0,3,,,No phone service | No | Yes
8,InternetService,object,7043,0,3,,,DSL | Fiber optic | No
9,OnlineSecurity,object,7043,0,3,,,No | Yes | No internet service


,column,dtype_after_load,non_null,missing,unique_non_null,min,max,sample_values
19,TotalCharges,Float64,7032,11,6530,18.8,8684.8,29.85 | 1889.5 | 108.15 | 1840.75 | 151.65


**Quality decision:** không xóa 11 khách hàng mới; việc xóa sẽ loại có hệ thống cohort `tenure=0`. Dùng constant imputation `0` trong fold vì tổng phí bằng 0 là giả định nghiệp vụ hợp lý cho account chưa phát sinh kỳ thanh toán.

## 2. Target audit — imbalance và accuracy trap

In [4]:
target = (
    data["Churn"].value_counts().rename_axis("Churn").to_frame("n")
).assign(rate=lambda frame: frame["n"] / len(data))
display(target)
ax = target["rate"].plot.bar(color=["#64748b", "#dc2626"], rot=0)
ax.set(title="Target distribution", ylabel="Share of customers")
ax.bar_label(ax.containers[0], fmt="%.1f")
plt.show()
print(f"Always-No accuracy: {(data['Churn'] == 'No').mean():.1%}; churn recall: 0%")

,n,rate
Churn,,
No,5174,0.73463
Yes,1869,0.26537


Always-No accuracy: 73.5%; churn recall: 0%


C:\Users\thanh\AppData\Local\Temp\ipykernel_24460\1740354800.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Accuracy khoảng 73,5% có thể đạt bằng cách không phát hiện churner nào, nên không dùng để chọn model. AP được chọn làm primary; ROC-AUC là secondary; Brier/log loss đo chất lượng xác suất; Recall/Lift@budget đo quyết định retention.

## 3. Numeric distributions và target relationship

In [5]:
numeric = numeric_summary(data)
display(numeric)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for column, axis in zip(["tenure", "MonthlyCharges", "TotalCharges"], axes):
    for churn, color in [("No", "#64748b"), ("Yes", "#dc2626")]:
        data.loc[data["Churn"].eq(churn), column].plot.hist(
            bins=25, density=True, alpha=0.45, ax=axis, color=color, label=churn
        )
    axis.set_title(column)
    axis.legend(title="Churn")
fig.tight_layout()
plt.show()

,feature,non_churn_n,non_churn_mean,non_churn_median,churn_n,churn_mean,churn_median,standardized_mean_difference_churn_minus_non_churn
0,tenure,5174,37.569965,38.000,1869,17.979133,10.00,-0.892829
1,MonthlyCharges,5174,61.265124,64.425,1869,74.441332,79.65,0.469507
2,TotalCharges,5163,2555.344141,1683.600,1869,1531.796094,703.55,-0.482463


C:\Users\thanh\AppData\Local\Temp\ipykernel_24460\1974337675.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Kiểm tra cả median/IQR trong bảng, không chỉ mean. `tenure` và `TotalCharges` phản ánh thời gian tồn tại account nên có thể là proxy lifecycle; chúng không chứng minh contract hay mức phí gây churn.

## 4. Categorical association, uncertainty và multiple testing

Bảng dùng chi-square, Cramér's V (effect size), Benjamini–Hochberg q-value và Wilson 95% CI theo level. Q-value giúp kiểm soát false discoveries khi khảo sát nhiều cột; effect size quan trọng hơn việc chỉ báo p-value.

In [6]:
rates = categorical_summary(data)
associations = rates[["feature", "cramers_v", "chi2_p_value", "chi2_q_value_bh"]].drop_duplicates()
display(associations.sort_values("cramers_v", ascending=False))
display(rates.loc[rates["feature"].isin(["Contract", "InternetService", "PaymentMethod"])])

,feature,cramers_v,chi2_p_value,chi2_q_value_bh
34,Contract,0.410116,5.863038e-258,9.380861e-257
16,OnlineSecurity,0.347400,2.661150e-185,2.128920e-184
25,TechSupport,0.342916,1.443084e-180,7.696448e-180
13,InternetService,0.322455,9.571788e-160,3.828715e-159
39,PaymentMethod,0.303359,3.682355e-140,1.178353e-139
19,OnlineBackup,0.292316,2.079759e-131,5.546025e-131
22,DeviceProtection,0.281580,5.505219e-122,1.258336e-121
31,StreamingMovies,0.230951,2.667757e-82,5.335514e-82
28,StreamingTV,0.230502,5.528994e-82,9.829324e-82
37,PaperlessBilling,0.191498,4.073355e-58,6.517367e-58


,feature,level,n,churners,churn_rate,ci95_low,ci95_high,chi2_p_value,cramers_v,chi2_q_value_bh
13,InternetService,DSL,2421,459,0.189591,0.174473,0.205692,9.571788e-160,0.322455,3.828715e-159
14,InternetService,Fiber optic,3096,1297,0.418928,0.401659,0.436397,9.571788e-160,0.322455,3.828715e-159
15,InternetService,No,1526,113,0.074050,0.061954,0.088285,9.571788e-160,0.322455,3.828715e-159
34,Contract,Month-to-month,3875,1655,0.427097,0.411602,0.442736,5.863038e-258,0.410116,9.380861e-257
35,Contract,One year,1473,166,0.112695,0.097543,0.129862,5.863038e-258,0.410116,9.380861e-257
36,Contract,Two year,1695,48,0.028319,0.021425,0.037345,5.863038e-258,0.410116,9.380861e-257
39,PaymentMethod,Bank transfer (automatic),1544,258,0.167098,0.149321,0.186529,3.682355e-140,0.303359,1.178353e-139
40,PaymentMethod,Credit card (automatic),1522,232,0.152431,0.135249,0.171363,3.682355e-140,0.303359,1.178353e-139
41,PaymentMethod,Electronic check,2365,1071,0.452854,0.432885,0.472976,3.682355e-140,0.303359,1.178353e-139
42,PaymentMethod,Mailed check,1612,308,0.191067,0.172618,0.210985,3.682355e-140,0.303359,1.178353e-139


In [7]:
top = associations.nlargest(8, "cramers_v").sort_values("cramers_v")
ax = top.plot.barh(x="feature", y="cramers_v", legend=False, color="#2563eb")
ax.set(title="Strongest univariate categorical associations", xlabel="Cramér's V")
plt.show()

C:\Users\thanh\AppData\Local\Temp\ipykernel_24460\688771944.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Cohort hypothesis — tenure × contract

In [8]:
tenure_band = pd.cut(
    data["tenure"], [-1, 6, 12, 24, 48, np.inf],
    labels=["0–6", "7–12", "13–24", "25–48", "49+"]
)
cohorts = (
    data.assign(tenure_band=tenure_band)
    .groupby(["tenure_band", "Contract"], observed=True)["churn_flag"]
    .agg(n="size", churn_rate="mean")
    .reset_index()
)
display(cohorts)
heatmap = cohorts.pivot(index="tenure_band", columns="Contract", values="churn_rate")
fig, ax = plt.subplots(figsize=(7, 4))
image = ax.imshow(heatmap, cmap="YlOrRd", vmin=0, vmax=0.7, aspect="auto")
ax.set_xticks(range(len(heatmap.columns)), heatmap.columns, rotation=20)
ax.set_yticks(range(len(heatmap.index)), heatmap.index)
ax.set(xlabel="Contract", ylabel="Tenure band", title="Observed churn rate by cohort")
fig.colorbar(image, ax=ax, label="Churn rate")
plt.show()

,tenure_band,Contract,n,churn_rate
0,0–6,Month-to-month,1413,0.552017
1,0–6,One year,39,0.102564
2,0–6,Two year,29,0.000000
3,7–12,Month-to-month,581,0.419966
4,7–12,One year,85,0.105882
5,7–12,Two year,39,0.000000
6,13–24,Month-to-month,737,0.377205
7,13–24,One year,197,0.081218
8,13–24,Two year,90,0.000000
9,25–48,Month-to-month,802,0.329177


C:\Users\thanh\AppData\Local\Temp\ipykernel_24460\925503593.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Leakage and prediction-time availability audit

| Field | Available at snapshot? | Decision | Risk |
|---|---:|---|---|
| `customerID` | Yes | identifier only, never a feature | memorization/privacy |
| account/service/contract/payment fields | Assumed yes | include | assumption must be checked in real warehouse |
| `TotalCharges` | Assumed yes | include; in-fold imputation | billing cutoff/skew |
| `Churn` / `churn_flag` | No | target only | direct leakage |

IBM sample không có event timestamp. Vì vậy không thể chứng minh out-of-time validity hay feature freshness; production replacement phải dùng point-in-time joins và time split.

## 7. EDA conclusions → experiment hypotheses

1. Dummy prior và always-No là hai no-skill references bắt buộc.
2. Logistic + OHE là baseline mạnh, dễ giải thích; tree boosting chỉ thắng nếu vượt complexity gate.
3. Không resample mặc định; class weights là ablation vì có thể làm calibration xấu đi.
4. Audit riêng `gender` và `SeniorCitizen`, đồng thời chạy Logistic bỏ hai trường này; không gọi đây là fairness proof.
5. Báo Recall/Lift ở budget 5/10/20/30%, calibration và 15 error cases.
6. Business value chỉ là sensitivity scenarios; muốn đo tác động giữ chân cần randomized treatment/control và uplift modeling.

Báo cáo EDA đã materialize nằm tại `reports/eda/EDA_REPORT.md`; chạy lại bằng `python -m telco_churn.eda`.